# MuCoCo RQ4 Second Order Results Aggregation

This notebook is used to aggregate the results for MuCoCo second order experiments. The experiment results are stored in MuCoCo_results/MuCoCo_experiment_results/second_order from the project directory. The results are reflected in Table XII (MUCOCO’s Scalability to multiple (2) mutations vs. atomic mutations).

In [1]:
import os
import sys
import pandas as pd
from typing import Tuple, Dict

In [2]:
curr_dir = os.getcwd()
parent_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(parent_dir)
sys.path.append(proj_dir)

In [3]:
from utility.data_log_functions import DataLogHelper

/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def standardize_two_df(df1: pd.DataFrame, df2: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    common_ids = set(df1["task_id"]) & set(df2["task_id"])
    if not common_ids:
        print("⚠️ No matching task_ids found between the two DataFrames.")
        return df1.iloc[0:0], df2.iloc[0:0]  # return empty aligned frames

    df1_filtered = df1[df1["task_id"].isin(common_ids)].copy()
    df2_filtered = df2[df2["task_id"].isin(common_ids)].copy()

    df1_filtered = df1_filtered.drop_duplicates(subset=["task_id"], keep="first")
    df2_filtered = df2_filtered.drop_duplicates(subset=["task_id"], keep="first")

    df1_filtered = df1_filtered.sort_values("task_id").reset_index(drop=True)
    df2_filtered = df2_filtered.sort_values("task_id").reset_index(drop=True)

    return df1_filtered, df2_filtered

In [5]:
def compare_multiple_code_generation_logs(res_dir: str, filter: Tuple[str] = (), anti_filter: Tuple[str] = ()):
    
    if filter is None:
        csv_logs = [f for f in os.listdir(res_dir) if (os.path.isfile(os.path.join(res_dir, f)) and f.endswith(".csv"))]
    else:
        csv_logs = [f for f in os.listdir(res_dir) if (
            os.path.isfile(os.path.join(res_dir, f)) and 
            f.endswith(".csv") and 
            all(sub in f for sub in filter)) and
            all(sub not in f for sub in anti_filter)
            ]

    log_file_names = [csv_file_name.replace('.csv', '') for csv_file_name in csv_logs]

    results_df = pd.DataFrame(columns=log_file_names, index = log_file_names)
    for file_name in log_file_names:
        results_df.loc[file_name, file_name] = float('nan')

    while len(csv_logs) > 0:
        log1_file_name = csv_logs.pop()
        for log2_file_name in csv_logs:
            log1_file_path = os.path.join(res_dir, log1_file_name)
            log2_file_path = os.path.join(res_dir, log2_file_name)

            log1 = pd.read_csv(log1_file_path)
            log2 = pd.read_csv(log2_file_path) 
            print(log1_file_name, log2_file_name)
            log1, log2 = standardize_two_df(log1, log2)
            log1_inconsistencies, log2_inconsistencies = DataLogHelper.compare_code_generation_dataframe_results(log1=log1, log2=log2)

            results_df.loc[log1_file_name.replace('.csv', ''), log2_file_name.replace('.csv', '')] = log1_inconsistencies
            results_df.loc[log2_file_name.replace('.csv', ''), log1_file_name.replace('.csv', '')] = log2_inconsistencies

    return results_df

In [6]:
def clean_up_csv_name(file_name: str)-> str:
    mutation_type = file_name.split("shot_")[-1]
    if "_" in mutation_type:
        mutation = mutation_type.replace("_", " ").title()
        return mutation
    return mutation_type.capitalize()

In [7]:
def obtain_category(log_name:str) -> str | None:

    second_order_mutation = {
        "Second Order": [
            "for2while_random",
            "for2while_constant_unfold",
            "constant_unfold_random"
        ]    
    }

    for cat, mut in second_order_mutation.items():
        for m in mut:
            if m in log_name:
                return cat
        
    else:
        return "Atomic"


In [8]:
def compare_logs_against_no_mutation(
        res_dir: str, 
        task: str,
        benchmark: str, 
        filter: Tuple[str] = (), 
        anti_filter: Tuple[str] = ()
    ):
    
    if filter is None:
        csv_logs = [f for f in os.listdir(res_dir) if (os.path.isfile(os.path.join(res_dir, f)) and f.endswith(".csv"))]
    else:
        csv_logs = [f for f in os.listdir(res_dir) if (
            os.path.isfile(os.path.join(res_dir, f)) and 
            f.endswith(".csv") and 
            all(sub in f for sub in filter)) and
            all(sub not in f for sub in anti_filter)
            ]

    csv_logs.sort()
    target_log_name = [l for l in csv_logs if "no_mutation" in l][-1]
    csv_logs.pop(csv_logs.index(target_log_name))
    target_log_path = os.path.join(res_dir, target_log_name)
    target_log = pd.read_csv(target_log_path)

    results_df = pd.DataFrame()

    total_inconsistencies = 0
    total_questions = 0
    total_success = 0
    total_answered = 0

    category_dict = {}
    mutation_dict = {}

    for log_name in csv_logs:
        # print(log_name)

        log_category = obtain_category(log_name)
                
        log2_file_path = os.path.join(res_dir, log_name)
        log2 = pd.read_csv(log2_file_path) 

        inconsistency_dict = DataLogHelper.compare_code_generation_dataframe_results(log1=target_log, log2=log2, task = task, benchmark = benchmark)

        # Metrics for model inconsistency calculation
        mutation_inconsistencies = inconsistency_dict['total_inconsistencies']
        mutation_cumulative_inc_dist = inconsistency_dict['cumulative_inconsistency_distance']
        mutation_questions = inconsistency_dict['total_inconsistency_comparisons']

        # Metrics for model accuracy calculation
        no_mutation_successes = inconsistency_dict['log1_success']
        no_mutation_answered = inconsistency_dict['log1_total_answered']
        mutation_successes = inconsistency_dict['log2_success']
        mutation_answered = inconsistency_dict['log2_total_answered']

        # Metrics for direction calculation
        mutation_incorrect_dir = inconsistency_dict['incorrect_dir']
        mutation_invalid_dir = inconsistency_dict['invalid_dir']    

        # Adding results into the dataframe
        cleaned_mutation_name = clean_up_csv_name(log_name.replace('.csv', ''))
        results_df.loc[cleaned_mutation_name, "Inconsistency Score"] = f"{mutation_inconsistencies}/{mutation_questions} ({round(mutation_inconsistencies*100/mutation_questions, 2)}%)"

        if 'No Mutation' in results_df.index:
            pass
        else:
            results_df.loc['No Mutation', "Inconsistency Score"] = "N/A"
            results_df.loc['No Mutation', "Model Accuracy"] = f"{no_mutation_successes}/{no_mutation_answered} ({round(no_mutation_successes*100/no_mutation_answered, 2)}%)"
        results_df.loc[cleaned_mutation_name, "Model Accuracy"] = f"{mutation_successes}/{mutation_answered} ({round(mutation_successes*100/mutation_answered, 2)}%)"
        
        if total_success == 0:
            total_success += no_mutation_successes
        
        if total_answered == 0:
            total_answered += no_mutation_answered

        if 'model_ensemble' in log_name.lower() or "ensemble" not in log_name.lower():
            total_inconsistencies += mutation_inconsistencies
            total_questions += mutation_questions
            total_success += mutation_successes
            total_answered += mutation_answered

        
        if log_category:
            d: Dict = category_dict.get(log_category, {})
            d['total_inconsistencies'] = d.get('total_inconsistencies', 0) + mutation_inconsistencies
            d['total_questions'] = d.get('total_questions', 0) + mutation_questions
            d['total_success'] = d.get('total_success', 0) + mutation_successes
            d['total_answered'] = d.get('total_answered', 0) + mutation_answered
            category_dict[log_category] = d


        # adding results in mutation_dict, with the mutation name as key
        mutation_dict[cleaned_mutation_name] = {
            'total_inconsistencies': mutation_inconsistencies,
            'total_questions': mutation_questions,
            'total_success': mutation_successes,
            'total_answered': mutation_answered
        }
    
    results_df = pd.concat([
        results_df[results_df.index.str.lower().str.contains("no mutation")],

        results_df[
            ~results_df.index.str.lower().str.contains("ensemble") &
            ~results_df.index.str.lower().str.contains("no mutation")
        ],

        results_df[results_df.index.str.lower().str.contains("ensemble")]
    ])

    ## Adding aggregated second order results and atomic results
    for key, mut_dict in category_dict.items():
        mut_inconsistencies = mut_dict['total_inconsistencies']
        mut_questions = mut_dict['total_questions']
        mut_success = mut_dict['total_success']
        mut_answered = mut_dict['total_answered']
        results_df.loc[f"{key} Results", "Inconsistency Score"] = f"{mut_inconsistencies}/{mut_questions} ({round(mut_inconsistencies*100/mut_questions, 2)})"
        results_df.loc[f"{key} Results", "Model Accuracy"] = f"{mut_success}/{mut_answered} ({round(mut_success*100/mut_answered, 2)}%)"

        mutation_dict[f"{key} Results"] = mut_dict


    results_df.loc["Aggregated Results", "Inconsistency Score"] = f"{total_inconsistencies}/{total_questions} ({round(total_inconsistencies*100/total_questions, 2)})"
    results_df.loc["Aggregated Results", "Model Accuracy"] = f"{total_success}/{total_answered} ({round(total_success*100/total_answered, 2)}%)"

    return [
        results_df, 
        category_dict, 
        ]

In [9]:
current_dir = os.getcwd()
proj_dir = os.path.abspath(os.path.join(current_dir, "..",))

def obtain_benchmark_task_csv(benchmark: str, task: str) -> pd.DataFrame:

    final_df = pd.DataFrame()  # start with an empty DataFrame

    res_dir = os.path.join(proj_dir, f"MuCoCo_experiment_results/second_order/{task}/")
    try:
        res, category_dict = compare_logs_against_no_mutation(
            task = task,
            benchmark = benchmark,
            res_dir=res_dir, 
            filter=(benchmark, )
        )

    except FileNotFoundError:
        print(f"{res_dir} does not exist.")

    res_df = pd.DataFrame(res)

    if final_df.empty:
        final_df = res_df
    else:
        final_df = pd.concat([final_df, res_df], axis=1)

    # final_df.to_csv("combined_results.csv", index=True, header=True)
    return final_df, category_dict


In [10]:
from tqdm import tqdm
import copy


tasks = {
    'input_prediction': ['HumanEval', "CruxEval"],
    'output_prediction': ['HumanEval', "CruxEval"],
    'mcq_inconsistency': ['CodeMMLU'],
}

task_dict = {}
overall_dict = {}
all_benchmark_dict = {}
dfs = []

def combine_two_dictionaries(d1: dict, d2: dict) -> dict:
    # make it PURE (return a new merged dict)
    out = copy.deepcopy(d1)
    for k, inner2 in d2.items():
        if k not in out:
            out[k] = copy.deepcopy(inner2)             
        else:
            out[k] += inner2
    return out

task_df = pd.DataFrame()

for task, benchmarks in tqdm(tasks.items()):
    # Dictionary for storing results to aggregate by task
    task_d = {}

    print(f"Aggregating for {task} logs")
    for benchmark in benchmarks:

        print(f"Working on {benchmark} now...")
        final_df, aggregated_dict = obtain_benchmark_task_csv(benchmark, task)

        final_df = final_df.rename(columns={
            "Inconsistency Score": f"{task}_{benchmark}_inconsistency",
            "Model Accuracy": f"{task}_{benchmark}_accuracy"
        })

        if task_df.empty:
            task_df = final_df
        else:
            task_df = pd.concat([task_df, final_df], axis=1)

        benchmark_dict = {}

        for mut_cat, mut_cat_dict in aggregated_dict.items():
            # make a NEW dict here instead of aliasing res_dir
            if not benchmark_dict.get(mut_cat, None):
                benchmark_dict[mut_cat] = mut_cat_dict.copy()
            else:
                for key, val in mut_cat_dict.items():
                    benchmark_dict[mut_cat][key] += val

        d1 = task_d.get(task, {})

        if not d1:
            task_d[task] = copy.deepcopy(mut_cat_dict)
        else:
            task_d[task] = combine_two_dictionaries(d1, mut_cat_dict)    
    
    task_dict[task] = task_d[task]



  0%|          | 0/3 [00:00<?, ?it/s]

Aggregating for input_prediction logs
Working on HumanEval now...
Working on CruxEval now...


 33%|███▎      | 1/3 [00:03<00:06,  3.27s/it]

Aggregating for output_prediction logs
Working on HumanEval now...
Working on CruxEval now...


100%|██████████| 3/3 [00:06<00:00,  1.76s/it]

Aggregating for mcq_inconsistency logs
Working on CodeMMLU now...


100%|██████████| 3/3 [00:06<00:00,  2.13s/it]


# MuCoCo Results Aggregated Across Tasks

In [11]:

order = [
    "No Mutation", 
    "Random",
    "Constant Unfold",
    "For2while",
    "Constant Unfold Random",
    "For2While Random",
    "For2While Constant Unfold",
    "Atomic Results",
    "Second Order Results",
    "Aggregated Results"
]

task_df = task_df.reindex(order)

for idx in task_df.index:
    inc = 0
    comparisons = 0
    acc = 0
    answered = 0
    for col in task_df.columns:
        val  = task_df.loc[idx, col]
        if val == "N/A" or isinstance(val, float):
            continue
        num = val.split("(")[0].split('/')
        numerator = int(num[0])
        denominator = int(num[1])
        if 'inconsistency' in col and 'mcq_inconsistency' not in col:
            inc += numerator
            comparisons += denominator
        else:
            acc += numerator
            answered += denominator
    try:
        task_df.loc[idx, "Aggregated Inc."] = f"{inc}/{comparisons} ({round(inc*100/comparisons,2)}%)"
    except ZeroDivisionError:
        task_df.loc[idx, "Aggregated Inc."] = "N/A"

    task_df.loc[idx, "Aggregated Acc."] = f"{acc}/{answered} ({round(acc*100/answered,2)}%)"


print(task_df.to_string())

                          input_prediction_HumanEval_inconsistency input_prediction_HumanEval_accuracy input_prediction_CruxEval_inconsistency input_prediction_CruxEval_accuracy output_prediction_HumanEval_inconsistency output_prediction_HumanEval_accuracy output_prediction_CruxEval_inconsistency output_prediction_CruxEval_accuracy mcq_inconsistency_CodeMMLU_inconsistency mcq_inconsistency_CodeMMLU_accuracy      Aggregated Inc.      Aggregated Acc.
No Mutation                                                    N/A                    750/995 (75.38%)                                     N/A                   646/800 (80.75%)                                       N/A                      588/995 (59.1%)                                      N/A                    430/799 (53.82%)                                      N/A                    114/133 (85.71%)                  N/A   2528/3722 (67.92%)
Random                                              97/995 (9.75%)                    749/995 